# Parte 3 – Cruce por ubigeo y análisis

Juntamos los decretos de la Parte 1 con la lluvia de la Parte 2 usando el ubigeo departamental del INEI, y con eso respondemos la pregunta del trabajo: ¿el Estado declara emergencia donde más llueve?


In [1]:
import os
import pandas as pd
import plotly.express as px

### Paso 1. Tabla de ubigeos

Creamos `datos/ubigeos.csv` con los 25 ubigeos departamentales oficiales del INEI. Los escribimos entre comillas para que sean texto desde el principio.

In [2]:
ubigeos = pd.DataFrame({
    "ubigeo": ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12", "13",
               "14", "15", "16", "17", "18", "19", "20", "21", "22", "23", "24", "25"],
    "departamento": ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho", "Cajamarca",
                     "Callao", "Cusco", "Huancavelica", "Huánuco", "Ica", "Junín", "La Libertad",
                     "Lambayeque", "Lima", "Loreto", "Madre de Dios", "Moquegua", "Pasco", "Piura",
                     "Puno", "San Martín", "Tacna", "Tumbes", "Ucayali"],
})
ubigeos.to_csv("datos/ubigeos.csv", index=False, encoding="utf-8-sig")
print("Filas:", len(ubigeos))
ubigeos.head()

Filas: 25


,ubigeo,departamento
0,01,Amazonas
1,02,Áncash
2,03,Apurímac
3,04,Arequipa
4,05,Ayacucho


### Paso 2 · Leer el archivo dos veces

In [3]:
ubigeos_como_numero = pd.read_csv("datos/ubigeos.csv", encoding="utf-8-sig")
ubigeos_como_texto = pd.read_csv("datos/ubigeos.csv", encoding="utf-8-sig", dtype={"ubigeo": str})

print("Lectura normal:", ubigeos_como_numero["ubigeo"].dtype,
      "| con dtype str:", ubigeos_como_texto["ubigeo"].dtype)

comparacion = pd.DataFrame({
    "lectura_normal": ubigeos_como_numero["ubigeo"],
    "con_dtype_str": ubigeos_como_texto["ubigeo"],
    "departamento": ubigeos_como_texto["departamento"],
})
comparacion.head(10)

Lectura normal: int64 | con dtype str: object


,lectura_normal,con_dtype_str,departamento
0,1,01,Amazonas
1,2,02,Áncash
2,3,03,Apurímac
3,4,04,Arequipa
4,5,05,Ayacucho
5,6,06,Cajamarca
6,7,07,Callao
7,8,08,Cusco
8,9,09,Huancavelica
9,10,10,Huánuco


**¿Por qué el ubigeo se lee como texto?**

Con la lectura normal, pandas cree que el ubigeo es un número y le borra el cero de la izquierda: «01» pasa a ser 1 y «09» pasa a ser 9. Pero el ubigeo es un **código**, no una cantidad: no tiene sentido sumarlo ni promediarlo. Si lo dejamos como número, el 1 ya no cruza con el «01» de otra tabla, y tampoco se puede relacionar con los ubigeos de provincia o de distrito, que empiezan con los mismos dos dígitos (el distrito «010101» pertenece al departamento «01»). Por eso lo leemos siempre con `dtype={"ubigeo": str}`.

### Paso 3. Agregar el ubigeo a las dos tablas

Unimos cada tabla con la de ubigeos por el nombre del departamento. Es el único momento en que usamos el nombre; el cruce entre decretos y lluvia lo hacemos después por ubigeo. Si algún nombre estuviera escrito distinto (una tilde, un espacio), ese departamento se quedaría sin ubigeo, y por eso lo revisamos.

In [4]:
decretos_dep = pd.read_csv("datos/decretos_por_departamento.csv", encoding="utf-8-sig")
lluvias_dep = pd.read_csv("datos/lluvias_por_departamento.csv", encoding="utf-8-sig")

decretos_con_ubigeo = decretos_dep.merge(ubigeos_como_texto, on="departamento", how="left")
lluvias_con_ubigeo = lluvias_dep.merge(ubigeos_como_texto, on="departamento", how="left")

In [5]:
sin_ubigeo_decretos = decretos_con_ubigeo.loc[decretos_con_ubigeo["ubigeo"].isna(), "departamento"]
sin_ubigeo_lluvias = lluvias_con_ubigeo.loc[lluvias_con_ubigeo["ubigeo"].isna(), "departamento"]

print("Decretos sin ubigeo:", list(sin_ubigeo_decretos))
print("Lluvias sin ubigeo:", list(sin_ubigeo_lluvias))

Decretos sin ubigeo: []
Lluvias sin ubigeo: []


In [6]:
correcciones = {"Ancash": "Áncash"}   # según lo que salga
lluvias_dep["departamento"] = lluvias_dep["departamento"].str.strip().replace(correcciones)
lluvias_con_ubigeo = lluvias_dep.merge(ubigeos_como_texto, on="departamento", how="left")
print("Lluvias sin ubigeo:", list(lluvias_con_ubigeo.loc[lluvias_con_ubigeo["ubigeo"].isna(), "departamento"]))

Lluvias sin ubigeo: []


**Resultado de la verificación**

Las dos listas salieron vacías desde la primera vez: en la Parte 1 usamos la lista oficial de departamentos, y en la Parte 2 los nombres de Wikipedia ya venían con la misma ortografía (Áncash, Junín, San Martín...) y el Callao se agregó con ese nombre. Los 25 departamentos de cada tabla tienen su ubigeo.

### Paso 4. Cruce por ubigeo

Juntamos lluvia y decretos por el ubigeo, no por el nombre. Partimos de la tabla de lluvias, que tiene los 25 departamentos, y usamos `how="left"`: así ningún departamento desaparece del cruce. Si un departamento no tuviera decretos, quedaría vacío, y lo convertimos en 0 con `fillna(0)`.

In [7]:
tabla_final = lluvias_con_ubigeo.merge(
    decretos_con_ubigeo[["ubigeo", "declaratorias", "prorrogas"]],
    on="ubigeo",    # por código, no por nombre
    how="left",     # se quedan los 25 departamentos de la tabla de lluvias
)

# Un departamento sin decretos quedaría vacío (NaN): eso significa 0
tabla_final["declaratorias"] = tabla_final["declaratorias"].fillna(0).astype(int)
tabla_final["prorrogas"] = tabla_final["prorrogas"].fillna(0).astype(int)

columnas = ["ubigeo", "departamento", "capital", "latitud", "longitud",
            "lluvia_total_mm", "dias_lluvia_fuerte", "declaratorias", "prorrogas"]
tabla_final = tabla_final[columnas].sort_values("ubigeo").reset_index(drop=True)
tabla_final

,ubigeo,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte,declaratorias,prorrogas
0,01,Amazonas,Chachapoyas,-6.23169,-77.86903,427.9,0,8,4
1,02,Áncash,Huaraz,-9.52614,-77.52869,1415.9,11,8,6
2,03,Apurímac,Abancay,-13.63426,-72.88380,820.0,3,7,2
3,04,Arequipa,Arequipa,-16.39899,-71.53747,439.4,3,8,2
4,05,Ayacucho,Ayacucho,-13.16380,-74.22345,500.8,1,8,4
5,06,Cajamarca,Cajamarca,-7.16378,-78.50027,875.7,2,7,2
6,07,Callao,Callao,-12.05162,-77.13452,54.0,0,2,0
7,08,Cusco,Cusco,-13.53188,-71.96701,801.3,4,9,6
8,09,Huancavelica,Huancavelica,-12.78691,-74.97298,930.4,4,8,2
9,10,Huánuco,Huánuco,-9.92882,-76.23989,617.0,4,8,5


In [8]:
print("Filas:", len(tabla_final))
print("Ubigeos repetidos:", tabla_final["ubigeo"].duplicated().sum())
print("Departamentos con 0 declaratorias:", (tabla_final["declaratorias"] == 0).sum())
print("Departamentos con 0 prórrogas:", list(tabla_final.loc[tabla_final["prorrogas"] == 0, "departamento"]))
print("Total de declaratorias:", tabla_final["declaratorias"].sum(),
      "| total de prórrogas:", tabla_final["prorrogas"].sum())

Filas: 25
Ubigeos repetidos: 0
Departamentos con 0 declaratorias: 0
Departamentos con 0 prórrogas: ['Callao', 'Moquegua']
Total de declaratorias: 157 | total de prórrogas: 71


**¿Aparecen los ceros?**

La tabla final tiene 25 filas. En nuestra temporada ningún departamento se quedó sin declaratorias (el mínimo es 2, en el Callao y Tumbes), pero sí hay ceros en las prórrogas: el Callao y Moquegua. Los totales (157 declaratorias y 71 prórrogas) son los mismos de la Parte 1, así que el cruce no perdió ni duplicó ningún departamento.

### Paso 5 · Guardar la tabla final

In [9]:
tabla_final.to_csv("datos/tabla_final.csv", index=False, encoding="utf-8-sig")

# Miramos el archivo como texto: el ubigeo debe decir 01, no 1
with open("datos/tabla_final.csv", encoding="utf-8-sig") as archivo:
    for numero_linea in range(3):
        print(archivo.readline().strip())

ubigeo,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte,declaratorias,prorrogas
01,Amazonas,Chachapoyas,-6.23169,-77.86903,427.9,0,8,4
02,Áncash,Huaraz,-9.52614,-77.52869,1415.9,11,8,6


###  Paso 6. Gráficos

El primer gráfico pone la lluvia de la temporada en el eje X y las declaratorias en el eje Y, con el nombre de cada departamento. Si el Estado declarara emergencia donde más llueve, los puntos subirían de izquierda a derecha. El segundo muestra los 10 departamentos con más declaratorias; a igual número de declaratorias, los ordenamos por prórrogas.

In [10]:
fig_dispersion = px.scatter(
    tabla_final,
    x="lluvia_total_mm",
    y="declaratorias",
    text="departamento",
    hover_data=["capital", "dias_lluvia_fuerte", "prorrogas"],
    title="¿Se declara emergencia donde más llueve? Enero a mayo de 2025",
    labels={"lluvia_total_mm": "Lluvia acumulada en la capital (mm)",
            "declaratorias": "Declaratorias de emergencia por lluvias"},
    template="plotly_white",
)
fig_dispersion.update_traces(textposition="top center", marker={"size": 10, "color": "#2a6f97"})
fig_dispersion.show()

In [11]:
# A igual número de declaratorias, se ordena por prórrogas
top10 = tabla_final.sort_values(["declaratorias", "prorrogas"], ascending=False).head(10)

fig_barras = px.bar(
    top10,
    x="departamento",
    y="declaratorias",
    hover_data=["prorrogas", "lluvia_total_mm"],
    title="Los 10 departamentos con más declaratorias por lluvias (enero a mayo de 2025)",
    labels={"departamento": "", "declaratorias": "Declaratorias de emergencia"},
    template="plotly_white",
    color_discrete_sequence=["#2a6f97"],
)
fig_barras.show()

In [12]:
os.makedirs("graficos", exist_ok=True)
fig_dispersion.write_html("graficos/dispersion_lluvia_declaratorias.html", include_plotlyjs="cdn")
fig_barras.write_html("graficos/top10_declaratorias.html", include_plotlyjs="cdn")

### Paso 7 · Datos para responder

In [13]:
# Pregunta 1: más declaratorias (keep="all" muestra los empates)
print(tabla_final.nlargest(3, "declaratorias", keep="all")[["departamento", "declaratorias", "prorrogas"]])

# Desempate opcional: a igual número de declaratorias, más prórrogas
ranking = tabla_final.sort_values(["declaratorias", "prorrogas"], ascending=False)
print(ranking[["departamento", "declaratorias", "prorrogas"]].head(3))

# Más lluvia en la capital
print(tabla_final.nlargest(3, "lluvia_total_mm")[["departamento", "capital", "lluvia_total_mm", "dias_lluvia_fuerte"]])

    departamento  declaratorias  prorrogas
7          Cusco              9          6
0       Amazonas              8          4
1         Áncash              8          6
3       Arequipa              8          2
4       Ayacucho              8          4
8   Huancavelica              8          2
9        Huánuco              8          5
14          Lima              8          3
15        Loreto              8          2
  departamento  declaratorias  prorrogas
7        Cusco              9          6
1       Áncash              8          6
9      Huánuco              8          5
     departamento           capital  lluvia_total_mm  dias_lluvia_fuerte
15         Loreto           Iquitos           2019.9                  28
16  Madre de Dios  Puerto Maldonado           1751.3                  30
1          Áncash            Huaraz           1415.9                  11


In [14]:
# Pregunta 2: comparamos cada departamento con la mediana de lluvia y de declaratorias
mediana_lluvia = tabla_final["lluvia_total_mm"].median()
mediana_declaratorias = tabla_final["declaratorias"].median()
print("Mediana de lluvia:", mediana_lluvia, "mm | mediana de declaratorias:", mediana_declaratorias)

mucha_lluvia_pocas = tabla_final[(tabla_final["lluvia_total_mm"] > mediana_lluvia)
                                 & (tabla_final["declaratorias"] < mediana_declaratorias)]
poca_lluvia_muchas = tabla_final[(tabla_final["lluvia_total_mm"] < mediana_lluvia)
                                 & (tabla_final["declaratorias"] > mediana_declaratorias)]

print("\nMucha lluvia y pocas declaratorias:")
print(mucha_lluvia_pocas[["departamento", "lluvia_total_mm", "declaratorias"]])
print("\nPoca lluvia y muchas declaratorias:")
print(poca_lluvia_muchas[["departamento", "lluvia_total_mm", "declaratorias"]])

Mediana de lluvia: 699.1 mm | mediana de declaratorias: 7.0

Mucha lluvia y pocas declaratorias:
     departamento  lluvia_total_mm  declaratorias
16  Madre de Dios           1751.3              3
18          Pasco            874.2              6
20           Puno           1074.4              6
24        Ucayali           1309.0              6

Poca lluvia y muchas declaratorias:
   departamento  lluvia_total_mm  declaratorias
0      Amazonas            427.9              8
3      Arequipa            439.4              8
4      Ayacucho            500.8              8
9       Huánuco            617.0              8
14         Lima             41.1              8


In [15]:
# Opcional: correlación por rangos (Spearman), va de -1 a 1
rango_lluvia = tabla_final["lluvia_total_mm"].rank()
rango_declaratorias = tabla_final["declaratorias"].rank()
print("Correlación de Spearman:", round(rango_lluvia.corr(rango_declaratorias), 2))

Correlación de Spearman: 0.24


In [16]:
# Pregunta 3: se cuenta por decreto (cada declaratoria una sola vez)
decretos = pd.read_csv("datos/decretos_lluvias.csv", encoding="utf-8-sig")
solo_declaratorias = decretos[decretos["tipo"] == "declara"]

print("Declaratorias por lluvias:", len(solo_declaratorias))
print(solo_declaratorias["motivo"].value_counts())
print((solo_declaratorias["motivo"].value_counts(normalize=True) * 100).round(1))

Declaratorias por lluvias: 11
motivo
impacto de daños     9
peligro inminente    2
Name: count, dtype: int64
motivo
impacto de daños     81.8
peligro inminente    18.2
Name: proportion, dtype: float64


### Paso 8 · Respuestas en celdas de texto

Son cinco celdas de texto. Los datos ya están puestos, calculados con los CSV de main. Antes de pegarlas, comprueben que su notebook da los mismos números y redacten el texto con sus palabras: se evalúa que las conclusiones sean de ustedes.

**1. ¿Cuáles fueron los 3 departamentos con más declaratorias? ¿Coinciden con los 3 donde más llovió?**

Cusco fue el departamento con más declaratorias (9). Le siguen ocho departamentos empatados con 8: Amazonas, Áncash, Arequipa, Ayacucho, Huancavelica, Huánuco, Lima y Loreto. Por el empate no hay un tercer lugar único; si desempatamos por número de prórrogas, el top 3 es Cusco, Áncash y Huánuco.

Los tres con más lluvia en su capital fueron Loreto (Iquitos, 2 019.9 mm), Madre de Dios (Puerto Maldonado, 1 751.3 mm) y Áncash (Huaraz, 1 415.9 mm). Coinciden solo en parte: Loreto y Áncash están en el grupo de 8 declaratorias, pero Madre de Dios tuvo solo 3, y Cusco, el de más declaratorias, quedó en el puesto 12 de 25 en lluvia (801.3 mm).

**2. ¿Hay departamentos con mucha lluvia y pocas declaratorias, o al revés?**

Sí. Comparando cada departamento con la mediana (699.1 mm de lluvia y 7 declaratorias):

- **Mucha lluvia y pocas declaratorias:** Madre de Dios (1 751.3 mm y 3 declaratorias), Ucayali (1 309.0 mm y 6), Puno (1 074.4 mm y 6) y Pasco (874.2 mm y 6).
- **Poca lluvia y muchas declaratorias:** Lima (41.1 mm en Huacho y 8), Amazonas (427.9 mm y 8), Arequipa (439.4 mm y 8), Ayacucho (500.8 mm y 8) y Huánuco (617.0 mm y 8).

Razones posibles:

- **Decretos en bloque:** 8 de las 11 declaratorias abarcan entre 14 y 22 departamentos a la vez, así que muchos departamentos suman lo mismo aunque en cada uno haya llovido muy distinto. Por eso ocho departamentos empatan con 8.
- **Lo que es normal en cada lugar:** en la selva (Madre de Dios, Ucayali) llueve mucho todos los años y la gente, las casas y las vías están preparadas para eso.
- **Dónde medimos:** la lluvia es la de la capital, pero los distritos declarados suelen estar en otras provincias y a otra altura. Lima se midió en Huacho y Arequipa en su ciudad capital, las dos en zonas secas, mientras que las emergencias suelen estar en la sierra.
- **Vulnerabilidad y capacidad de gestión:** lo que convierte la lluvia en desastre son las viviendas en quebradas o riberas, el drenaje y la pobreza; además, la declaratoria la piden los gobiernos regionales con informes técnicos.

**3. ¿Qué porcentaje fue por peligro inminente y qué porcentaje por impacto de daños?**

De las 11 declaratorias por lluvias de la temporada, 2 (18.2 %) fueron por peligro inminente, es decir, antes del daño: el DS 007-2025-PCM (10 de enero) y el DS 033-2025-PCM (12 de marzo). Las otras 9 (81.8 %) fueron por impacto de daños, cuando el daño ya había ocurrido, y las 7 prórrogas también. En 2025 el Estado usó la declaratoria sobre todo para responder a daños ya ocurridos, más que para prepararse ante un peligro anunciado: la gestión fue más reactiva que preventiva.

**4. Limitaciones**

- **La capital no representa a todo el departamento:** medimos la lluvia en un solo punto, y los distritos declarados pueden estar lejos y a otra altura. La lluvia de Lima, por ejemplo, se midió en Huacho, en la costa.
- **Datos de modelo, que parecen exagerar en los Andes:** Open-Meteo entrega un reanálisis (un modelo sobre una grilla de varios kilómetros), no la lectura de una estación del SENAMHI. Huaraz (1 415.9 mm) y Huancayo (1 349.1 mm) salen con más lluvia que Pucallpa (1 309.0 mm), en plena Amazonía, lo que sugiere que el modelo sobreestima la lluvia de los valles andinos.
- **Contamos decretos, no su tamaño:** un decreto con un distrito pesa lo mismo que uno con 200; no medimos distritos, población ni daños.
- **Los departamentos salen del título:** si un título nombra mal un lugar, o un distrito se llama como un departamento, el conteo cambia.
- **El buscador de gob.pe puede no traer todos los decretos**, y algunos decretos de enero pueden responder a lluvias de diciembre de 2024, que no medimos.

**Conclusión**

Solo en parte. La relación entre la lluvia de cada capital y el número de declaratorias es positiva pero débil (correlación de Spearman de 0.24). Loreto y Áncash, entre los tres con más lluvia, están en el grupo de 8 declaratorias, pero Madre de Dios, el segundo con más lluvia, tuvo solo 3, y Lima, con 41.1 mm en Huacho, tuvo 8. Cusco, el departamento con más declaratorias (9), quedó en el puesto 12 de lluvia. Las declaratorias llegaron en decretos que abarcan hasta 22 departamentos a la vez y, en 9 de 11 casos (81.8 %), después del daño: el Estado respondió a los daños más que a los milímetros de lluvia. Como medimos la lluvia en una sola ciudad por departamento y con datos de modelo, esta conclusión es aproximada.